In [1]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

In [2]:
import pandas as pd
from datasets import Dataset, ClassLabel
import numpy as np
import random
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from transformers import Trainer
from transformers import EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader

In [3]:
OUTPUT_MODEL_NAME = "synth_lora_model_distilclinicalbert"
CHECKPOINT_DIR = "checkpoints_distilclinicalbert"
TENSORBOARD_RUN_NAME = "synthetic_data_experiment_distilclinicalbert"

In [4]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

# Make constant variables.
MODEL_NAME = "nlpie/distil-clinicalbert"

In [5]:
# Load the dataset
df = pd.read_csv("./dataSyntheticAll.csv")
dataset = Dataset.from_pandas(df)

In [6]:
# Map dataset labels to 1s or 0s.
label_map = {
    "met": 0,
    "unmet": 1
}

dataset = dataset.map(lambda x: {"label": label_map[x["needs"]]})


label_feature = ClassLabel(names=["met", "unmet"])
dataset = dataset.cast_column("label", label_feature)

Map:   0%|          | 0/5783 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5783 [00:00<?, ? examples/s]

In [7]:
# Tokenize texts.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(example):
    return tokenizer(
        example["report"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tokenize)
 # Set PyTorch format. 
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/5783 [00:00<?, ? examples/s]

In [8]:
# Split the dataset. (80/10/10)
dataset = dataset.train_test_split(test_size=0.2, seed=RANDOM_STATE, stratify_by_column="label")

train_dataset = dataset["train"]
temp_dataset = dataset["test"]

temp_split = temp_dataset.train_test_split(test_size=0.5, seed=RANDOM_STATE, stratify_by_column="label")

val_dataset = temp_split["train"]
test_dataset = temp_split["test"]

# Check distributions.
def check_distribution(dataset, name):
    df = dataset.to_pandas()
    counts = df["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(train_dataset, "Train")
check_distribution(val_dataset, "Validation")
check_distribution(test_dataset, "Test")


Train distribution:
label
0    0.502162
1    0.497838
Name: proportion, dtype: float64
Validation distribution:
label
0    0.50173
1    0.49827
Name: proportion, dtype: float64
Test distribution:
label
0    0.502591
1    0.497409
Name: proportion, dtype: float64


In [9]:
# Confirm device for use. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
# Load the base model.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# Define LoRA config.
lora_config = LoraConfig(
    r=8, # LoRA attention dimension (rank)
    lora_alpha=16, # alpha for LoRA scaling
    target_modules=["attention.self.query", "attention.self.value", "attention.output.dense", "intermediate.dense"], # specific named modules to be replaced
    lora_dropout=0.05, # dropout probability for LoRA layers
    bias="none", # bias type
    task_type="SEQ_CLS" # what type of task (sequence classification)
)

# Attach LoRA to model. 
model = get_peft_model(model, lora_config)
model.to(device)
model.print_trainable_parameters()

Device: cuda


pytorch_model.bin:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpie/distil-clinicalbert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you ex

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

trainable params: 407,042 || all params: 66,191,620 || trainable%: 0.6149


In [10]:
# Set training arguments.
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    learning_rate=2e-5,
    num_train_epochs=50, # higher so early stopping can trigger
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard",
    run_name=TENSORBOARD_RUN_NAME,
    fp16=True
)

# Add early stopping.
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2
)

# Add metrics function.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Create trainer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks = [early_stopping],
    compute_metrics=compute_metrics
)

In [11]:
# Train the model.
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.282571,0.333163,0.877163,0.835913,0.937500,0.883797
2,0.241339,0.331993,0.904844,0.881967,0.934028,0.907251
3,0.257299,0.326671,0.906574,0.872611,0.951389,0.910299
4,0.264586,0.293186,0.920415,0.890323,0.958333,0.923077
5,0.219159,0.301202,0.927336,0.918367,0.937500,0.927835
6,0.207707,0.292132,0.929066,0.897106,0.968750,0.931553
7,0.193757,0.318867,0.930796,0.892405,0.979167,0.933775
8,0.180847,0.305470,0.927336,0.889241,0.975694,0.930464


TrainOutput(global_step=9256, training_loss=0.2386879817926379, metrics={'train_runtime': 5465.1534, 'train_samples_per_second': 42.323, 'train_steps_per_second': 10.585, 'total_flos': 2474314757406720.0, 'train_loss': 0.2386879817926379, 'epoch': 8.0})

In [12]:
# Save the fine-tuned model.
model.save_pretrained(OUTPUT_MODEL_NAME)
tokenizer.save_pretrained(OUTPUT_MODEL_NAME)

('synth_lora_model_distilclinicalbert/tokenizer_config.json',
 'synth_lora_model_distilclinicalbert/tokenizer.json')

In [13]:
# Move model to eval mode.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

loader = DataLoader(test_dataset, batch_size=32)

preds = []
labels = []

with torch.no_grad():
    for batch in loader:
        # Move all tensors to the same device as the model
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=1)

        preds.extend(predictions.cpu().numpy())   # move to CPU for metrics
        labels.extend(batch_labels.cpu().numpy())

preds = np.array(preds)
labels = np.array(labels)

# Confusion Matrix
cm = confusion_matrix(labels, preds)
print("--------------- Confusion Matrix ---------------")
print(cm)

# Classification Report
report = classification_report(labels, preds)
print("--------------- Classification Report ---------------")
print(report)

--------------- Confusion Matrix ---------------
[[265  26]
 [  7 281]]
--------------- Classification Report ---------------
              precision    recall  f1-score   support

           0       0.97      0.91      0.94       291
           1       0.92      0.98      0.94       288

    accuracy                           0.94       579
   macro avg       0.94      0.94      0.94       579
weighted avg       0.94      0.94      0.94       579

